# EZStats AI Worker — run on a Colab GPU

**The point of this notebook:** use Colab's GPU for the heavy steps while the code
stays on your PC, and get the results back automatically — *without re-uploading
the repo every time you change a line*.

## How it avoids re-uploading

| What | How it gets to Colab | When |
|---|---|---|
| **Code** (`src/`, run scripts) | `git pull` from GitHub | every run — seconds, incremental |
| **Models** (`artifacts/`, 315 MB) | Google Drive | **once** |
| **Videos** (`data/raw/`) | Google Drive | once per new video |
| **Results** (`outputs/<run>/`) | written to Drive -> Drive for Desktop syncs them to your PC | every run |

So the edit loop is: **edit locally -> `git push` -> re-run cell 4 -> run.**
No zipping, no manual upload, no `Compress-Archive` backslash problems.

---

## One-time setup (do this once, then never again)

1. **Install Google Drive for Desktop** on Windows and sign in. This is what makes
   results appear on your PC automatically.
2. In Drive, create a folder **`ezstats`** with two subfolders:
   - `ezstats/artifacts/` — copy your local `artifacts/` into it (315 MB: the
     player, ball and pitch models)
   - `ezstats/data/raw/`  — copy the match videos you want to process
3. Runtime -> Change runtime type -> **GPU (T4 is fine)** -> Save.
4. Run the cells below in order.

> Colab is Linux, so `run_pipeline_v2.ps1` will not run here. `run_pipeline_v2.py`
> is the portable runner with identical steps and flags — that is what this
> notebook calls, so local and Colab runs stay in sync.

### 1. Confirm we actually got a GPU
If this prints `No GPU`, fix the runtime type before continuing — everything below will still work, just at CPU speed.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "No GPU")
import torch; print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

### 2. Mount Drive
Authorise when prompted. This is where the models, videos and results live.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/ezstats')
assert DRIVE.exists(), f"Create {DRIVE} in Drive first (see the setup steps above)."
print("Drive OK:", DRIVE)
for p in sorted(DRIVE.iterdir()):
    print("  ", p.name)

### 3. Get the code — clone the first time, pull every time after

This is the cell that replaces re-uploading. After you `git push` from your PC,
just re-run this cell and Colab has your latest code in a couple of seconds.

If the repo is private, generate a GitHub personal access token and use
`https://<TOKEN>@github.com/MattyKKS/EZStatsAIWorker.git` as the URL.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/MattyKKS/EZStatsAIWorker.git"
BRANCH   = "test"            # change to the branch you are working on
REPO     = Path('/content/EZStatsAIWorker')

if REPO.exists():
    print(subprocess.run(["git", "fetch", "--all"], cwd=REPO, text=True, capture_output=True).stdout)
    print(subprocess.run(["git", "checkout", BRANCH], cwd=REPO, text=True, capture_output=True).stdout)
    print(subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO, text=True, capture_output=True).stdout)
else:
    print(subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], text=True, capture_output=True).stdout)

print(subprocess.run(["git", "log", "--oneline", "-3"], cwd=REPO, text=True, capture_output=True).stdout)

### 4. Install dependencies

~2-3 minutes on a fresh runtime. Colab already ships torch with CUDA, so we do not
touch it. `sports` is Roboflow's helper package (pitch config, ByteTrack helpers)
and is not on PyPI, hence the git install.

In [ ]:
%pip install -q ultralytics supervision umap-learn transformers sentencepiece
%pip install -q git+https://github.com/roboflow/sports.git
%pip install -q -e /content/EZStatsAIWorker
print("deps installed")

### 5. Link models and videos from Drive

Symlinks, not copies — no waiting for 315 MB to duplicate. `artifacts/` and
`data/` are gitignored, which is exactly why they come from Drive instead of git.

In [ ]:
import os
from pathlib import Path

REPO  = Path('/content/EZStatsAIWorker')
DRIVE = Path('/content/drive/MyDrive/ezstats')

for name in ("artifacts", "data"):
    link, target = REPO / name, DRIVE / name
    if not target.exists():
        raise SystemExit(f"Missing {target} in Drive — upload it (see setup step 2).")
    if link.is_symlink() or link.exists():
        if link.is_symlink():
            link.unlink()
        else:
            raise SystemExit(f"{link} exists and is not a symlink — remove it first.")
    os.symlink(target, link)
    print(f"{name} -> {target}")

print("\nVideos available:")
for v in sorted((DRIVE / 'data' / 'raw').glob('*.mp4')):
    print(f"  {v.name}  ({v.stat().st_size/1e6:.0f} MB)")

### 6. Run the pipeline

Outputs are written to Colab's **local** disk first. Drive is slow for the many
small files a run produces (player crops especially), so we write locally and copy
the finished folder to Drive in the next cell.

> ### Keep the video ON here
> `stats_video.mp4` is how you actually **check the events with your own eyes** —
> whether that "pass" was a pass, whether the goal got detected. A report full of
> numbers you cannot verify is worth very little.
>
> On CPU that render is ~30 min of a ~38 min run, which is exactly why it is worth
> moving to a GPU. **On Colab you render — that is the whole point of being here.**
> `--skip-video` exists only for fast local threshold tuning, when you are counting
> events in the JSON and not yet looking at footage. Leave `SKIP_VIDEO = False`.

Flags:
- `--source-fps 60` for the Messi clip — it scales the frame-based tracking knobs.
- `--goal-frame N` marks the shot nearest frame N as a goal.
- `--skip-video` — local tuning only; see the warning above.

In [ ]:
VIDEO      = "data/raw/BrightonGoal.mp4"
SOURCE_FPS = 25
SKIP_VIDEO = False
GOAL_FRAME = None      # e.g. 1050

import subprocess, sys
from pathlib import Path
REPO = Path('/content/EZStatsAIWorker')

cmd = [sys.executable, "run_pipeline_v2.py", "--video", VIDEO, "--source-fps", str(SOURCE_FPS)]
if SKIP_VIDEO:            cmd.append("--skip-video")
if GOAL_FRAME is not None: cmd += ["--goal-frame", str(GOAL_FRAME)]

proc = subprocess.Popen(cmd, cwd=REPO, text=True, bufsize=1,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
run_dir = None
for line in proc.stdout:
    print(line, end="")
    if line.startswith("Run dir:"):
        run_dir = line.split(":", 1)[1].strip()
proc.wait()
print("\nrun_dir =", run_dir)

### 7. Copy the results back

Writes the run folder to `ezstats/outputs/` in Drive. With Drive for Desktop
running, it appears on your PC by itself — no download step.

The big intermediate `processed_video.mp4` is skipped (it is redundant with
`stats_video.mp4`, and the backend never serves it).

In [ ]:
import shutil
from pathlib import Path

REPO  = Path('/content/EZStatsAIWorker')
DRIVE = Path('/content/drive/MyDrive/ezstats')

src = (REPO / run_dir) if run_dir else None
assert src and src.exists(), f"Run dir not found: {src}"
dst = DRIVE / 'outputs' / src.name
dst.mkdir(parents=True, exist_ok=True)

for item in sorted(src.iterdir()):
    if item.name == "processed_video.mp4":
        print(f"  skip {item.name} (redundant intermediate)")
        continue
    target = dst / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)
    print(f"  {item.name}")

print(f"\nCopied to {dst}")
print("Drive for Desktop will sync this to your PC under  Google Drive\\ezstats\\outputs\\")

### 8. Quick look at the result

Sanity-check the numbers before you go back to your PC.

In [ ]:
import json
from collections import Counter
from pathlib import Path

rep = Path('/content/EZStatsAIWorker') / run_dir / 'match_report_merged.json'
if not rep.exists():
    rep = rep.with_name('match_report.json')
d = json.loads(rep.read_text())

print(rep.name)
print("  players   :", len(d['players']), Counter(p.get('team_id') for p in d['players']))
print("  possession:", d.get('possession'))
print("  events    :", len(d['events']), Counter(e['type'] for e in d['events']))
for e in d['events']:
    print(f"    {e['type']:<13} t={e['time_s']:<7} {e.get('actor_label')} -> {e.get('target_label')}")

---
## The loop from here

1. Edit code on your PC.
2. `git add -A && git commit -m "..." && git push`
3. In Colab: re-run **cell 3** (`git pull`) and **cell 6** (run).
4. Results land in Drive and sync to your PC.

## Honest caveats

- **Colab sessions are ephemeral.** `/content` is wiped when the runtime recycles.
  Anything you care about must go to Drive (cell 7). Re-running cells 3-5 rebuilds
  the environment in ~3 minutes.
- **Idle disconnects.** Colab drops idle sessions; keep the tab open for long runs.
- **Drive sync is not instant** for large files — a 260 MB `stats_video.mp4` takes
  a while to appear on the PC.
- **Uncommitted local changes do not reach Colab.** The `git reset --hard` in cell 3
  is deliberate — it guarantees Colab matches your pushed branch exactly — but it
  means anything you forgot to push is simply not there.
- **`processed_video.mp4` is skipped** on copy-back by design.